In [1]:
import pandas as pd

Data loading

In [2]:
TRAIN_PATH = "/Users/david/Documents/data/jusbrasil/manual/with_body/splits/train.parquet"
VAL_PATH = "/Users/david/Documents/data/jusbrasil/manual/with_body/splits/vali.parquet"
#TEST_PATH = "/Users/david/Documents/data/jusbrasil/manual/with_body/splits/small_test.parquet"
TEST_PATH = "/Users/david/Documents/data/jusbrasil/manual/with_body/splits/vali.parquet"

In [3]:
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

Prepocessing

In [4]:
def preprocess_llms(df):
   return df[["query","title","body","court","relevance"]]

In [5]:
train_df = preprocess_llms(train_df)
test_df = preprocess_llms(test_df)
val_df = preprocess_llms(val_df)

In [6]:
train_df['body'].str.len().describe()

count      9188.000000
mean      20108.203200
std       24784.232236
min           0.000000
25%        8397.500000
50%       14141.500000
75%       23782.750000
max      449686.000000
Name: body, dtype: float64

Evaluating by LLMs

In [7]:
from pathlib import Path
import os

from dotenv import load_dotenv

load_dotenv(Path("/Users/david/Documents/dev/llm-as-judge-for-ltr/.env"))

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "").strip()
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is missing. Put it in the project .env file.")

In [8]:
import os
import pandas as pd
from openai import OpenAI
from functools import cmp_to_key

# client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",    
#     api_key=os.getenv("OPENROUTER_API_KEY"),
# )


#Connect to Ollama
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [9]:
# MODEL = "qwen/qwen-2.5-72b-instruct"

#MODEL = "gemma2:2b"
#MODEL = "gemma2:9b"
# MODEL = "meta-llama/llama-3.3-70b-instruct"
MODEL = "llama3.1:8b"

In [10]:
CHAR_SIZE = 3000

In [11]:
import time

def format_doc(row, char_limit=3000):
    title = str(row.get("title", "Sem título"))
    court = str(row.get("court", "Tribunal Nao Informado"))
    body = str(row.get("body", ""))[:char_limit]
    return f"Title: {title}\nCourt: {court}\nText: {body}..."


def compare_documents(doc_a, doc_b, query_text):
    system_prompt = "You are an expert Brazilian legal information retrieval judge. Your task is to compare two documents and determine which one is more relevant to a user's search query."
    user_prompt = f"""Query: {query_text}

    Document A:
    {format_doc(doc_a, CHAR_SIZE)}

    Document B:
    {format_doc(doc_b, CHAR_SIZE)}

    Which document is more relevant to the query? Output exactly 'A' if document A is more relevant, 'B' if document B is more relevant. Do not provide any explanation or extra text."""

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.0,
            max_tokens=2,
        )
        winner = response.choices[0].message.content.strip().upper()

        #time.sleep(3.1)
        
        if "A" in winner:
            return 1
        if "B" in winner:
            return -1
        return 0
    except Exception as e:
        print(f"Error comparing documents: {e}")
        return 0

In [12]:
from functools import cmp_to_key

from tqdm.auto import tqdm


def run_pairwise_pilot(test_df):
    results = []
    grouped = list(test_df.groupby("query"))

    for query_text, group in tqdm(grouped, total=len(grouped), desc="Ranking queries"):
        docs = group.to_dict(orient="records")

        def custom_compare(a, b):
            return compare_documents(a, b, query_text)

        docs.sort(key=cmp_to_key(custom_compare), reverse=True)

        for rank, doc in enumerate(docs, start=1):
            doc["llm_rank"] = rank
            results.append(doc)

    return pd.DataFrame(results)



/Users/david/Library/Caches/pypoetry/virtualenvs/llm-as-judge-for-ltr-UctXQ_0n-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
df_llm_ranked = run_pairwise_pilot(test_df)
print("Ranking complete.")

Ranking queries: 100%|██████████| 109/109 [4:41:25<00:00, 154.91s/it] 

Ranking complete.


In [14]:
import numpy as np

def ndcg_at_k(labels_sorted_by_score, labels_ideal, k):
    k = min(k, len(labels_sorted_by_score))
    predicted = labels_sorted_by_score[:k]
    ideal = np.sort(labels_ideal)[::-1][:k]
    discounts = np.log2(np.arange(1, k + 1) + 1)
    dcg = np.sum((np.power(2.0, predicted) - 1) / discounts)
    idcg = np.sum((np.power(2.0, ideal) - 1) / discounts)
    return 0.0 if idcg == 0.0 else dcg / idcg

ks = [1, 3, 5, 10]
rows = []

for query, g in df_llm_ranked.groupby("query"):
    y_q = g["relevance"].to_numpy(dtype=float)
    # rank 1 first (ascending), unlike LambdaMART which sorts score descending
    y_ranked = y_q[np.argsort(g["llm_rank"].to_numpy())]

    row = {"query": query, "num_docs": len(g)}
    for k in ks:
        row[f"ndcg@{k}"] = ndcg_at_k(y_ranked, y_q, k)
    rows.append(row)

metrics_llm = pd.DataFrame(rows)
for k in ks:
    print(f"LLM  NDCG@{k:<3} = {metrics_llm[f'ndcg@{k}'].mean():.4f}")

LLM  NDCG@1   = 0.6592
LLM  NDCG@3   = 0.6645
LLM  NDCG@5   = 0.6867
LLM  NDCG@10  = 0.7211


In [15]:
from pathlib import Path

RESULTS_PATH = Path("/Users/david/Documents/dev/llm-as-judge-for-ltr/notebooks/data/PoC_results.csv")


def append_poc_results(model, metrics_df, notes=""):
    row = {
        "model": model,
        "ndcg@1": round(metrics_df["ndcg@1"].mean(), 4),
        "ndcg@3": round(metrics_df["ndcg@3"].mean(), 4),
        "ndcg@5": round(metrics_df["ndcg@5"].mean(), 4),
        "ndcg@10": round(metrics_df["ndcg@10"].mean(), 4),
        "n_queries": len(metrics_df),
        "notes": notes,
    }
    new = pd.DataFrame([row])
    if RESULTS_PATH.exists():
        out = pd.concat([pd.read_csv(RESULTS_PATH), new], ignore_index=True)
    else:
        out = new
    out.to_csv(RESULTS_PATH, index=False)
    return out


In [16]:
append_poc_results(MODEL, metrics_llm, notes= f"pairwise LLM judge on small_test, char_size={CHAR_SIZE}")

,model,ndcg@1,ndcg@3,ndcg@5,ndcg@10,n_queries,notes
0,qwen/qwen-2.5-72b-instruct,0.7698,0.7733,0.8037,0.8328,30,pairwise LLM judge on small_test 3000 body size
1,openai/gpt-4o-mini,0.6365,0.7514,0.7823,0.8171,30,pairwise LLM judge on small_test 3000 body size
2,qwen/qwen-2.5-72b-instruct,0.6111,0.5927,0.6511,0.7332,30,pairwise LLM judge on 30000 body size
3,openai/gpt-4o-mini,0.5778,0.6508,0.6913,0.7730,30,pairwise LLM judge on small_test 30000 body sieze
4,openai/gpt-4o-2024-11-20,0.7698,0.7969,0.8029,0.8406,30,pairwise LLM judge on small_test 10000 body size
5,openai/gpt-4o-2024-11-20,0.8254,0.8494,0.8616,0.8719,30,pairwise LLM judge on small_test 3000 body size
6,google/gemma-4-26b-a4b-it,0.9032,0.9284,0.9217,0.9199,30,pairwise LLM judge on small_test 3000 body size
7,gemma2:2b,0.5667,0.5822,0.6542,0.7259,30,pairwise LLM judge on small_test 3000 body size
8,gemma2:9b,0.8667,0.8100,0.8240,0.8528,30,pairwise LLM judge on small_test 3000 body size
9,llama3.2:3b,0.5889,0.5917,0.6543,0.7325,30,pairwise LLM judge on small_test
